# Imports

In [12]:
import pandas as pd
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import os

import sqlite3

# Load data

In [ ]:
CSV_PATH = "../data/csv/processed/"
files    = os.listdir(CSV_PATH)
csvs     = sorted([file.split('_')[1].split('.')[0] for file in files])

historic_csvs = csvs[:-1]
train_csv = pd.concat([pd.read_csv(f"{CSV_PATH}products_{csv}.csv") for csv in historic_csvs], ignore_index=True)

latest_csv = csvs[-1]
test_csv = pd.read_csv(f"{CSV_PATH}products_{latest_csv}.csv")

In [18]:
df = pd.read_csv("../data/csv/mail_groceries.csv")

conn = sqlite3.connect("../data/sql/swipes.db")
labels = pd.read_sql_query("SELECT data_id, is_liked, is_superliked, is_passed FROM swipes", conn)
conn.close()

In [19]:
labels

,data_id,is_liked,is_superliked,is_passed
0,10862993,1,0,0
1,10876624,1,0,0
2,10834070,0,1,0
3,10848222,1,0,0
4,10820980,1,0,0
...,...,...,...,...
571,10834055,1,0,0
572,10863003,1,0,0
573,10848105,1,0,0
574,10834062,1,0,0


# Pre-processing 

In [ ]:
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [ ]:
passage = df['translated_product'].to_numpy()
passage_embeddings = model.encode(passage)

# query = ["cupcake", "ice cream", "toilet paper", "grapes"]
# query_embeddings = model.encode(query)

# cosine_scores = cos_sim(query_embeddings, passage_embeddings)
# print(cosine_scores)

# best_idx = cosine_scores.argmax(axis=1)
# print(passage[best_idx])

tensor([[0.5429, 0.5627, 0.7577, 0.4324, 0.5832, 0.5989, 0.4385, 0.6743, 0.5200,
         0.6149, 0.5195, 0.6013, 0.6535, 0.6141, 0.6535, 0.6343, 0.6931, 0.8572,
         0.5181, 0.5700],
        [0.4906, 0.5428, 0.6359, 0.4329, 0.5549, 0.5710, 0.4497, 0.5984, 0.5396,
         0.5422, 0.5159, 0.6055, 0.6778, 0.6686, 0.6778, 0.7179, 0.7683, 0.7084,
         0.4805, 0.5947],
        [0.6456, 0.6599, 0.7687, 0.4482, 0.7860, 0.9702, 0.5977, 0.6150, 0.6455,
         0.6161, 0.5531, 0.5504, 0.6133, 0.6076, 0.6133, 0.6521, 0.5200, 0.6330,
         0.5154, 0.5652],
        [0.5218, 0.5538, 0.5717, 0.4989, 0.5666, 0.5369, 0.4556, 0.5748, 0.5401,
         0.6229, 0.5649, 0.7028, 0.6604, 0.7074, 0.6604, 0.6610, 0.6893, 0.5894,
         0.5438, 0.6455]])
['Cake plate' 'Giant Blackcurrant Popsicles' 'Toilet paper' 'Orange soda']


In [ ]:
onehot = ['brand','category']

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), onehot)
])
X = preprocessor.fit_transform(df).toarray()



# Train/test split

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42)